In [ ]:
!pip -q install openai faiss-cpu pypdf numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 11.9 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import faiss
from pypdf import PdfReader
from google.colab import files
from openai import OpenAI
from getpass import getpass

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass("enter a api key:")
client = OpenAI()

enter a api key:··········


In [ ]:
uploaded = files.upload()

Saving 100 Java Interview Questions with Answers.pdf to 100 Java Interview Questions with Answers.pdf


In [ ]:
def read_pdfs(file_names):
    text = ""
    for file_name in file_names:
        reader = PdfReader(file_name)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

raw_text = read_pdfs(uploaded.keys())

def chunk_text(text, chunk_size=1200, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c.strip() for c in chunks if c.strip()]

chunks = chunk_text(raw_text)
print("Total chunks:",len(chunks))

Total chunks: 35


In [ ]:
def get_embeddings(text_list, model="text-embedding-3-small"):
    resp = client.embeddings.create(
        model=model,
        input=text_list
    )
    return [d.embedding for d in resp.data]

embeddings = get_embeddings(chunks)


In [ ]:
dim = len(embeddings[0])
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings, dtype="float32"))

In [ ]:
def ask_pdf(question, top_k=3, distance_threshold=1.2):
    # Embed user question
    q_emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    ).data[0].embedding

    # Search top chunks
    D, I = index.search(np.array([q_emb], dtype="float32"), top_k)

    # If not relevant enough -> reject
    if D[0][0] > distance_threshold:
        return "No content found"

    context = "\n\n".join([chunks[i] for i in I[0]])

    prompt = f"""
You must answer ONLY from the CONTEXT below.
If the answer is not clearly present in the CONTEXT, reply exactly:
No content found

CONTEXT:
{context}

QUESTION:
{question}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "Answer only from the given PDF context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    answer = response.choices[0].message.content.strip()
    return answer if answer else "No content found"

In [ ]:
while True:
    q = input("\nAsk a question (type 'exit' to stop): ")
    if q.lower() == "exit":
        break
    print("\nAnswer:", ask_pdf(q))


Ask a question (type 'exit' to stop): What is default switch case? Give example. 

Answer: In a switch statement, default case is executed when no other switch condition matches. Default case is an optional case. It can be declared only once all other switch cases have been coded.

Example:
```java
public class switchExample {
    int score = 4;
    public static void main(String args[]) {
        switch (score) {
            case 1:
                System.out.println("Score is 1");
                break;
            case 2:
                System.out.println("Score is 2");
                break;
            default:
                System.out.println("Default Case");
        }
    }
}
```
In the above example, when score is not 1 or 2, default case is used.

Ask a question (type 'exit' to stop):  What are Java Packages? What's the significance of packages?

Answer: In Java, package is a collection of classes and interfaces which are bundled together as they are related to each other.